# API Exploration

Starter notebook for exploring `sailing_traffic` directly from the API
https://data.gov.gr/datasets/sailing_traffic

In [11]:
from pathlib import Path

import pandas as pd
import requests

In [12]:
API_URL = 'https://data.gov.gr/api/v1/query/sailing_traffic'
DATE_FROM = '2026-03-16'
DATE_TO = '2026-03-16'

In [27]:
response = requests.get(
    API_URL,
    params={
        'date_from': DATE_FROM, 
        'date_to': DATE_TO
    },
    timeout=60,
)

response.raise_for_status()
payload = response.json()

In [23]:
pd.__version__

'2.3.3'

In [15]:
df = pd.DataFrame(payload)
df.head()

,departureportname,vehiclecount,date,arrivalport,departureport,passengercount,routecode,routecodenames,arrivalportname
0,ΧΑΝΙΑ,1,2026-03-16,ADL,CHQ,2,CHQADLPIR,ΧΑΝΙΑ-ΑΔΑΜΑΝΤΑΣ ΜΗΛΟΥ-ΠΕΙΡΑΙΑΣ,ΑΔΑΜΑΝΤΑΣ ΜΗΛΟΥ
1,ΗΡΑΚΛΕΙΟ,5,2026-03-16,ADL,HER,11,HERADLPIR,ΗΡΑΚΛΕΙΟ-ΑΔΑΜΑΝΤΑΣ ΜΗΛΟΥ-ΠΕΙΡΑΙΑΣ,ΑΔΑΜΑΝΤΑΣ ΜΗΛΟΥ
2,ΠΕΙΡΑΙΑΣ,9,2026-03-16,ADL,PIR,10,PIRADLCHQ,ΠΕΙΡΑΙΑΣ-ΑΔΑΜΑΝΤΑΣ ΜΗΛΟΥ-ΧΑΝΙΑ,ΑΔΑΜΑΝΤΑΣ ΜΗΛΟΥ
3,ΠΕΙΡΑΙΑΣ,30,2026-03-16,ADL,PIR,116,PIRADLHER,ΠΕΙΡΑΙΑΣ-ΑΔΑΜΑΝΤΑΣ ΜΗΛΟΥ-ΗΡΑΚΛΕΙΟ,ΑΔΑΜΑΝΤΑΣ ΜΗΛΟΥ
4,ΑΓΚΙΣΤΡΙ ΑΙΓΙΝΑΣ,2,2026-03-16,AEG,AGG,44,AGGAEGPIR,ΑΓΚΙΣΤΡΙ ΑΙΓΙΝΑΣ-ΑΙΓΙΝΑ-ΠΕΙΡΑΙΑΣ,ΑΙΓΙΝΑ


In [24]:
len(df.columns)

9

In [26]:
df.shape

(415, 9)

In [25]:
df.dtypes

departureportname    object
vehiclecount          int64
date                 object
arrivalport          object
departureport        object
passengercount        int64
routecode            object
routecodenames       object
arrivalportname      object
dtype: object

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 415 entries, 0 to 414
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   departureportname  415 non-null    object
 1   vehiclecount       415 non-null    int64 
 2   date               415 non-null    object
 3   arrivalport        415 non-null    object
 4   departureport      415 non-null    object
 5   passengercount     415 non-null    int64 
 6   routecode          415 non-null    object
 7   routecodenames     415 non-null    object
 8   arrivalportname    415 non-null    object
dtypes: int64(2), object(7)
memory usage: 29.3+ KB


In [28]:
top_routes = (
    df.groupby(['routecode', 'routecodenames'], dropna=False, as_index=False)[['passengercount', 'vehiclecount']]
    .sum()
    .sort_values('passengercount', ascending=False)
    .head(10)
)
top_routes

,routecode,routecodenames,passengercount,vehiclecount
19,IGOCFU,ΗΓΟΥΜΕΝΙΤΣΑ-ΚΕΡΚΥΡΑ,2045,790
0,AEGPIR,ΑΙΓΙΝΑ-ΠΕΙΡΑΙΑΣ,1500,269
8,CFUIGO,ΚΕΡΚΥΡΑ-ΗΓΟΥΜΕΝΙΤΣΑ,1357,514
62,PIRAEG,ΠΕΙΡΑΙΑΣ-ΑΙΓΙΝΑ,1281,229
44,KYLZTH,ΚΥΛΛΗΝΗ-ΖΑΚΥΝΘΟΣ,1240,446
61,PIRADLHER,ΠΕΙΡΑΙΑΣ-ΑΔΑΜΑΝΤΑΣ ΜΗΛΟΥ-ΗΡΑΚΛΕΙΟ,1164,275
31,KERTSO,ΚΕΡΑΜΩΤΗ-ΘΑΣΟΣ,908,384
25,JMKTINANDRAF,ΜΥΚΟΝΟΣ-ΤΗΝΟΣ-ΑΝΔΡΟΣ-ΡΑΦΗΝΑ,880,268
73,PIRPASJNXJTR,ΠΕΙΡΑΙΑΣ-ΠΑΡΟΣ-ΝΑΞΟΣ-ΘΗΡΑ,879,262
87,RAFANDTINJMK,ΡΑΦΗΝΑ-ΑΝΔΡΟΣ-ΤΗΝΟΣ-ΜΥΚΟΝΟΣ,877,332


In [21]:
top_departure_ports = (
    df.groupby('departureportname', dropna=False, as_index=False)[['passengercount', 'vehiclecount']]
    .sum()
    .sort_values('passengercount', ascending=False)
    .head(10)
)
top_departure_ports

,departureportname,passengercount,vehiclecount
62,ΠΕΙΡΑΙΑΣ,8068,2410
19,ΗΓΟΥΜΕΝΙΤΣΑ,2324,946
7,ΑΙΓΙΝΑ,1826,295
40,ΚΥΛΛΗΝΗ,1753,692
36,ΚΕΡΚΥΡΑ,1399,514
70,ΡΑΦΗΝΑ,964,365
35,ΚΕΡΑΜΩΤΗ,908,384
18,ΖΑΚΥΝΘΟΣ,834,298
21,ΗΡΑΚΛΕΙΟ,811,378
22,ΘΑΣΟΣ,773,319


In [20]:
daily_totals = (
    df.assign(date=pd.to_datetime(df['date']))
    .groupby('date', as_index=False)[['passengercount', 'vehiclecount']]
    .sum()
)
daily_totals

,date,passengercount,vehiclecount
0,2026-03-16,30165,9523
